In [0]:
df = spark.read.parquet("dbfs:/mnt/light/data/world_bank_global_postings/version=5/2025/07/09/18_00_44/*.parquet")

In [0]:
from pyspark.sql.functions import col, count, month, year, to_date, concat, lit, when, last_day, lpad, sum as _sum

# Filter for US & jobs posted after 2017-01-01
df_us = df.filter((col('nation_name') == "United States") & (col('posted') >= "2023-01-01"))

# Filter to only include naics code 21 for grey industry jobs\
df_us = df_us.filter((col('naics2') == 23))

# Compile the DataFrame with necessary transformations
new_df = (
    df_us.withColumn("month", lpad(month(col("posted")).cast("string"), 2, "0"))  # Ensures 01, 02, ..., 12
    .withColumn("year", year(col("posted")))
    .groupBy("year", "month", "laa_admin_area_2", "min_years_experience", "isco_08_1_name")
    .agg(
        count(when(col("employment_type") == 1, True)).alias("full_time_count"),
        count(when(col("employment_type") == 2, True)).alias("part_time_count"),
        count("*").alias("total_count")
    )
    .withColumnRenamed("laa_admin_area_2", "county_id")
    .withColumnRenamed("isco_08_1_name", "career_type")
    .withColumn("start_date", to_date(concat(col("year"), lit("-"), col("month"), lit("-01")), "yyyy-MM-dd"))
    .withColumn("end_date", last_day(col("start_date")))  # Ensure correct last day
)

# Display the final compiled DataFrame
display(new_df)

year,month,county_id,min_years_experience,career_type,full_time_count,part_time_count,total_count,start_date,end_date
2023,02,US12083,2,"Plant and Machine Operators, and Assemblers",1,0,1,2023-02-01,2023-02-28
2024,03,US12109,null,Technicians and Associate Professionals,6,0,6,2024-03-01,2024-03-31
2023,02,US51159,1,Craft and Related Trades Workers,2,0,2,2023-02-01,2023-02-28
2024,03,US6037,1,Managers,8,0,8,2024-03-01,2024-03-31
2023,02,US17031,2,Professionals,14,0,15,2023-02-01,2023-02-28
2024,03,US13121,null,Technicians and Associate Professionals,46,4,51,2024-03-01,2024-03-31
2024,07,US25017,10,Managers,6,0,6,2024-07-01,2024-07-31
2024,03,US39113,null,Professionals,11,0,11,2024-03-01,2024-03-31
2024,03,US51059,5,Professionals,19,0,20,2024-03-01,2024-03-31
2023,02,US15003,null,"Plant and Machine Operators, and Assemblers",4,0,4,2023-02-01,2023-02-28


In [0]:
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/construction_vacancies_data_2023_24"

# Append data as CSV (instead of Parquet)
new_df.write.format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .save(output_path)